In [35]:
# ==========================================================
# 1. PREPARACIÓN DE DATOS
# ==========================================================

import pandas as pd
import numpy as np
import os
from sklearn.metrics import root_mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.metrics import MeanAbsoluteError, MeanAbsolutePercentageError
from tensorflow.keras.optimizers import RMSprop
from sklearn.preprocessing import MinMaxScaler
from time import time


ruta = r"C:\Users\Ximena\DLPronostico\ARIMA\data.csv"
print (ruta)

C:\Users\Ximena\DLPronostico\ARIMA\data.csv


In [36]:
import tensorflow as tf

# Verificar si hay GPUs disponibles
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"¡GPU detectada! {len(gpus)} dispositivo(s) GPU disponibles:")
    for gpu in gpus:
        print(f"  - {gpu}")
    # Puedes verificar que una operación se ejecuta en la GPU
    with tf.device('/GPU:0'):
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[1.0, 1.0], [0.0, 1.0]])
        c = tf.matmul(a, b)
    print("\nOperación de ejemplo ejecutada en GPU:\n", c.numpy())
else:
    print("No se detectaron GPUs. Asegúrate de que el entorno de ejecución esté configurado para usar GPU.")


¡GPU detectada! 1 dispositivo(s) GPU disponibles:
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Operación de ejemplo ejecutada en GPU:
 [[1. 3.]
 [3. 7.]]


In [37]:
# ==========================================================
# 1. PREPARACIÓN DE DATOS
# ==========================================================

def preparar_datos(ruta_csv):

    #       orden temporal en el tiempo - serie de tiempo
    # --------------------------------------------------------------
    df = pd.read_csv(ruta_csv, parse_dates=["beginTimeSeconds"])
    df = df.sort_values("beginTimeSeconds")
    df = df.set_index("beginTimeSeconds")
    # --------------------------------------------------------------

    metricas = [
        "Free Memory",
        "Used Memory",
        "Free Disk",
        "Used Disk",
        "Disk read/s",
        "Disk write/s",
        "NetBytes In",
        "NetBytes Out",
        "NetPackets In",
        "NetPackets Out",
        "Rx packets",
        "Tx packets",
        "CPU percent",
        "Memory Used percent",
        "Disk Used percent",
        "Uptime",
    ]

    df = df[metricas]
    df = df.ffill()
    return df


df = preparar_datos(ruta)

In [38]:
## ==========================================================
# 2. SELECCIÓN DE MÉTRICAS
# ==========================================================


def seleccionar_metricas(df):
    print(
        "\nSeleccione las métricas a analizar (ej: 1,3,5 o nombre, separados por coma):\n"
    )
    columnas = list(df.columns)
    for i, col in enumerate(columnas, 1):
        print(f"{i}. {col}")
    seleccion = input("\nIngrese los números o nombres de las métricas: ")
    metricas = []
    for x in seleccion.split(","):
        x = x.strip()
        if x.isdigit():
            idx = int(x)
            if 1 <= idx <= len(columnas):
                metricas.append(columnas[idx - 1])
        elif x in columnas:
            metricas.append(x)
    if not metricas:
        print("No se seleccionó ninguna métrica. Se usará 'Free Memory' por defecto.")
        metricas = ["Free Memory"]
    print(f"\nMétricas seleccionadas: {metricas}")
    return metricas


metricas_seleccionadas = seleccionar_metricas(df)



Seleccione las métricas a analizar (ej: 1,3,5 o nombre, separados por coma):

1. Free Memory
2. Used Memory
3. Free Disk
4. Used Disk
5. Disk read/s
6. Disk write/s
7. NetBytes In
8. NetBytes Out
9. NetPackets In
10. NetPackets Out
11. Rx packets
12. Tx packets
13. CPU percent
14. Memory Used percent
15. Disk Used percent
16. Uptime



Ingrese los números o nombres de las métricas:  8



Métricas seleccionadas: ['NetBytes Out']


In [39]:
# ==========================================================
# GENERACIÓN DE FOLDS (PREQUENTIAL CROSS VALIDATION)
# ==========================================================

fold_size = 1512
total_len = len(df)

n_folds = 3  # numero de iteraciones
folds = []

# for i in range(4):
for i in range(n_folds):

    train_end = fold_size * (i + 1)
    test_start = train_end
    # test_end = min(train_end + fold_size, total_len)
    test_end = train_end + fold_size

    folds.append((0, train_end, test_start, test_end))

print("\nITERATIONS GENERADOS:")

for i, f in enumerate(folds):
    print(f"F{i} -> Train[0:{f[1]}] Test[{f[2]}:{f[3]}]")


ITERATIONS GENERADOS:
F0 -> Train[0:1512] Test[1512:3024]
F1 -> Train[0:3024] Test[3024:4536]
F2 -> Train[0:4536] Test[4536:6048]


In [40]:
# ==========================================================
# 3. DEFINICIÓN DEL MODELO LSTM
# ==========================================================


def root_mean_squared_error(y_true, y_pred):
    return tf.math.sqrt(tf.math.reduce_mean(tf.square(y_pred - y_true)))


def crear_modelo(input_shape):

    model = Sequential(
        [
            tf.keras.Input(shape=input_shape),
            LSTM(128, return_sequences=True),
            Dropout(0.1),
            LSTM(128, return_sequences=True),
            LSTM(64, return_sequences=False),
            Dense(16, activation="relu"),
            Dense(1, activation="linear"),
        ]
    )

    model.compile(
        optimizer=RMSprop(learning_rate=5e-5),
        loss=root_mean_squared_error,
        metrics=[MeanAbsoluteError(), MeanAbsolutePercentageError()],
    )

    return model



In [41]:
# ==========================================================
# 4. FUNCIÓN DE VENTANAS - modificado
# ==========================================================


def crear_ventanas(data, window_size=10):

    X, y = [], []

    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size])
        y.append(data[i + window_size])

    return np.array(X), np.array(y)


# ------------------------ SE AGREGO ESTO -----------------------------

# ==========================================================
# WALK-FORWARD RECURSIVE FORECAST
# ==========================================================


def recursive_walk_forward(
    model, serie_train_scaled, serie_test_scaled, scaler, window_size=10
):

    history = list(serie_train_scaled.flatten())

    predictions = []

    for i in range(len(serie_test_scaled)):

        input_window = np.array(history[-window_size:])
        input_window = input_window.reshape(1, window_size, 1)

        yhat = model.predict(input_window, verbose=0)[0][0]

        predictions.append(yhat)

        history.append(yhat)

    predictions = np.array(predictions).reshape(-1, 1)

    preds_inv = scaler.inverse_transform(predictions)

    return preds_inv



In [42]:
# ==========================================================
# 5. NÚMERO DE RUNS
# ==========================================================


def seleccionar_numero_runs():
    print("\nSeleccione el número de runs para el experimento:")
    opciones = [10, 30, 50]
    for idx, op in enumerate(opciones, 1):
        print(f"{idx}. {op} runs")
    seleccion = input(
        "Ingrese el número de opción (1, 2 o 3) o escriba el número de runs: "
    )
    try:
        if seleccion.strip().isdigit():
            val = int(seleccion)
            if val in opciones:
                n_runs = val
            elif val > 0:
                n_runs = val
            else:
                n_runs = 10
        else:
            idx = int(seleccion)
            if 1 <= idx <= len(opciones):
                n_runs = opciones[idx - 1]
            else:
                n_runs = 10
    except:
        n_runs = 10
    print(f"\nNúmero de runs seleccionado: {n_runs}")
    return n_runs


n_runs = seleccionar_numero_runs()

# ==========================================================
# FUNCIÓN FORECAST ONE-STEP
# ==========================================================


def forecast_one_step(model, X_test, scaler):
    y_pred = model.predict(X_test, verbose=0)
    y_pred_inv = scaler.inverse_transform(y_pred)
    return y_pred_inv


# ==========================================================
# FUNCIÓN FORECAST MULTI-STEP
# ==========================================================


def forecast_multi_step(
    model, serie_scaled, scaler, window_size=10, n_steps=5
):  # n es le numero de pasos a predecir

    input_seq = serie_scaled[-window_size:].reshape(1, window_size, 1)
    preds = []

    for _ in range(n_steps):
        next_pred = model.predict(input_seq, verbose=0)
        preds.append(next_pred[0, 0])

        next_pred_reshaped = next_pred.reshape(1, 1, 1)
        input_seq = np.concatenate([input_seq[:, 1:, :], next_pred_reshaped], axis=1)

    preds = np.array(preds).reshape(-1, 1)
    preds_inv = scaler.inverse_transform(preds)

    return preds_inv




Seleccione el número de runs para el experimento:
1. 10 runs
2. 30 runs
3. 50 runs


Ingrese el número de opción (1, 2 o 3) o escriba el número de runs:  10



Número de runs seleccionado: 10


In [43]:
import tensorflow as tf
print("GPUs disponibles:", tf.config.list_physical_devices('GPU'))


GPUs disponibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [44]:
# ==========================================================
# 6. EXPERIMENTO (sin data leakage) / modificado
# ==========================================================

print("\nSeleccione el modo de predicción:")
print("1. One-step (predicción paso a paso)")
print("2. Multi-step (acumula n pasos)")
modo = input("Ingrese 1 o 2: ").strip()

if modo == "2":
    n_steps = input("Ingrese el número de pasos para multi-step: ").strip()
    try:
        n_steps = int(n_steps)
        if n_steps < 1:
            n_steps = 5
    except:
        n_steps = 5
else:
    modo = "1"
    n_steps = None

resultados_runs = []
metrics_fold_rows = []  # ---- se añadio esto

for run in range(n_runs):
    print(f"\n================ RUN {run+1} ================")
    resultados_metricas = []

    for metrica in metricas_seleccionadas:
        print(f"\nProcesando métrica: {metrica}")

        serie = df[metrica].values.reshape(-1, 1)

        # División temporal PRIMERO
        # ----------------- SE CAMBIO ESTO ---------------------------------------------
        # total_len = len(serie)
        # train_end = int(total_len * 0.7) #ajustar a los a FOLDS este bloque
        # val_end = int(total_len * 0.85)

        # serie_train = serie[:train_end]
        # serie_val = serie[train_end:val_end]
        # serie_test = serie[val_end:]
        # ------------------------- POR ESTO -------------------------------------
        for fold_id, (train_start, train_end, test_start, test_end) in enumerate(folds):

            print(f"\nITERATION {fold_id}")

            serie_train = serie[train_start:train_end]
            serie_test = serie[test_start:test_end]

            # validation dentro del train
            val_split = int(len(serie_train) * 0.8)

            serie_val = serie_train[val_split:]
            serie_train = serie_train[:val_split]
            # --------------------------------------------------------------

            # Escalado SOLO aprende del TRAIN (sin leakage)
            # --------------------------------------------------------------
            scaler = MinMaxScaler()
            serie_train_scaled = scaler.fit_transform(serie_train)
            serie_val_scaled = scaler.transform(serie_val)
            serie_test_scaled = scaler.transform(serie_test)
            # --------------------------------------------------------------

            # Crear ventanas por conjunto
            X_train, y_train = crear_ventanas(serie_train_scaled, window_size=10)
            X_val, y_val = crear_ventanas(serie_val_scaled, window_size=10)
            X_test, y_test = crear_ventanas(serie_test_scaled, window_size=10)

            inicio = time()

            # Crear y entrenar el modelo
            # --------------------------------------------------------------
            model = crear_modelo((X_train.shape[1], 1))

            model.fit(
                X_train,
                y_train,
                validation_data=(X_val, y_val),
                epochs=250,
                batch_size=256,
                verbose=0,
                callbacks=[
                    tf.keras.callbacks.EarlyStopping(
                        monitor="val_loss", patience=2, restore_best_weights=True
                    )
                    # -------------------------------------------------------------
                ],
            )

            if modo == "2":
                serie_total_scaled = scaler.transform(serie)
                y_pred_inv = forecast_multi_step(
                    model, serie_total_scaled, scaler, window_size=10, n_steps=n_steps
                )
                y_test_inv = scaler.inverse_transform(y_test[-n_steps:])
            else:
                # ----------- SE CAMBIO ESTO -------------

                # y_pred_inv = forecast_one_step(model, X_test, scaler)
                # y_test_inv = scaler.inverse_transform(y_test)

                # -------------- POR ESTO -----------
                y_pred_inv = recursive_walk_forward(
                    model, serie_train_scaled, serie_test_scaled, scaler, window_size=10
                )

                y_test_inv = scaler.inverse_transform(serie_test_scaled)
                # ---------------------------------------

            rmse = float(root_mean_squared_error(y_test_inv, y_pred_inv).numpy())

            mae_metric = MeanAbsoluteError()
            mape_metric = MeanAbsolutePercentageError()

            mae_metric.update_state(y_test_inv, y_pred_inv)
            mape_metric.update_state(y_test_inv, y_pred_inv)

            mae = mae_metric.result().numpy()
            mape = mape_metric.result().numpy()

            tiempo = (time() - inicio) * 1000

            print(
                f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.4f} | Tiempo: {tiempo:.2f} ms"
            )

            # ---------------------- SE AGREGA ESTO PARA GUARDAR CADA METRICA ------------------------------
            # ==========================================================
            # GUARDAR MÉTRICAS POR RUN Y FOLD
            # ==========================================================

            excel_path = "metrics-fold.xlsx"

            # fila = pd.DataFrame({ --- antes
            metrics_fold_rows.append(
                {
                    "Metric": metrica,
                    "Run": run + 1,
                    "Iteration": fold_id,
                    "RMSE": rmse,
                    "MAE": mae,
                    "MAPE": mape,
                    "Time": tiempo,
                }
            )

            #if os.path.exists(excel_path):

            #   df_existente = pd.read_excel(excel_path)

            #    df_existente = df_existente[
             #       ~(
             #           (df_existente["Metric"] == metrica)
             #           & (df_existente["Run"] == run + 1)
             #           & (df_existente["Fold"] == fold_id)
             #       )
             #   ]

             #   df_actualizado = pd.concat([df_existente, fila], ignore_index=True)

             #   df_actualizado.to_excel(excel_path, index=False)

            #else:

                #fila.to_excel(excel_path, index=False)

            # ------------------------------ CIERRA BLOQUE -------------------------------------------------------

            resultados_metricas.append([rmse, mae, mape, tiempo])

        resultados_metricas = np.array(resultados_metricas)
        promedio_run = resultados_metricas.mean(axis=0)

        print("\nPROMEDIO DEL RUN")
        print("------------------------------------------------------")
        print(
            f"RMSE: {promedio_run[0]:.4f} | MAE: {promedio_run[1]:.4f} | MAPE: {promedio_run[2]:.4f} | Tiempo: {promedio_run[3]:.2f} ms"
        )

        resultados_runs.append(promedio_run)

# ==========================================================
# GUARDAR RESULTADOS POR FOLD
# ==========================================================

excel_path = "metrics-fold.xlsx"

df_metrics_fold = pd.DataFrame(metrics_fold_rows)

df_metrics_fold = df_metrics_fold.sort_values(
    ["Metric","Run","Iteration"]
)

if os.path.exists(excel_path):

    df_existente = pd.read_excel(excel_path)

    # eliminar solo las métricas que se recalcularon
    df_existente = df_existente[
        ~df_existente["Metric"].isin(metricas_seleccionadas)
    ]

    df_final = pd.concat([df_existente, df_metrics_fold], ignore_index=True)

else:

    df_final = df_metrics_fold


df_final = df_final.sort_values(["Metric","Run","Iteration"])

df_final.to_excel(excel_path, index=False)

print("\nmetrics-fold.xlsx actualizado correctamente")


Seleccione el modo de predicción:
1. One-step (predicción paso a paso)
2. Multi-step (acumula n pasos)


Ingrese 1 o 2:  2
Ingrese el número de pasos para multi-step:  5



================ RUN 1 ================

Procesando métrica: NetBytes Out

ITERATION 0
RMSE: 0.0003 | MAE: 0.0003 | MAPE: 0.0022 | Tiempo: 3294.98 ms

ITERATION 1
RMSE: 0.0081 | MAE: 0.0081 | MAPE: 0.0537 | Tiempo: 3395.54 ms

ITERATION 2
RMSE: 0.0295 | MAE: 0.0295 | MAPE: 0.1969 | Tiempo: 3810.95 ms

PROMEDIO DEL RUN
------------------------------------------------------
RMSE: 0.0126 | MAE: 0.0126 | MAPE: 0.0842 | Tiempo: 3500.49 ms

================ RUN 2 ================

Procesando métrica: NetBytes Out

ITERATION 0
RMSE: 0.0000 | MAE: 0.0000 | MAPE: 0.0000 | Tiempo: 3498.93 ms

ITERATION 1
RMSE: 0.0007 | MAE: 0.0007 | MAPE: 0.0049 | Tiempo: 3158.47 ms

ITERATION 2
RMSE: 0.0031 | MAE: 0.0031 | MAPE: 0.0210 | Tiempo: 4336.56 ms

PROMEDIO DEL RUN
------------------------------------------------------
RMSE: 0.0013 | MAE: 0.0013 | MAPE: 0.0086 | Tiempo: 3664.65 ms

================ RUN 3 ================

Procesando métrica: NetBytes Out

ITERATION 0
RMSE: 0.0006 | MAE: 0.0006 | MAPE:

In [45]:
# ==========================================================
# 7. ESTADÍSTICAS FINALES
# ==========================================================

resultados_runs = np.array(resultados_runs)

metric_names = ["RMSE", "MAE", "MAPE", "Training time"]

print("\n================ RESULTADOS FINALES ================",  {metrica})

for i, metric in enumerate(metric_names):
    values = resultados_runs[:, i]

    mean_val = np.mean(values)
    sd_val = np.std(values, ddof=1)
    min_val = np.min(values)
    max_val = np.max(values)

    print(f"\n{metric}")
    print("--------------------------------------------------")
    print(f"mean : {mean_val:.4f}")
    print(f"SD   : {sd_val:.4f}")
    print(f"min  : {min_val:.4f}")
    print(f"max  : {max_val:.4f}")



================ RESULTADOS FINALES ================ {'NetBytes Out'}

RMSE
--------------------------------------------------
mean : 0.0067
SD   : 0.0034
min  : 0.0013
max  : 0.0126

MAE
--------------------------------------------------
mean : 0.0067
SD   : 0.0034
min  : 0.0013
max  : 0.0126

MAPE
--------------------------------------------------
mean : 0.0450
SD   : 0.0229
min  : 0.0086
max  : 0.0842

Training time
--------------------------------------------------
mean : 3589.7100
SD   : 196.4434
min  : 3341.2152
max  : 3971.4714


In [46]:
# ==========================================================
# 8. GUARDAR RESULTADOS EN EXCEL (ACTUALIZABLE)
# ==========================================================

import openpyxl
import os

excel_path = "resultados_lstm.xlsx"

# Crear estructura de resultados por indicador
for metrica in metricas_seleccionadas:

    resultados_dict = {}

    for i, metric in enumerate(metric_names):
        values = resultados_runs[:, i]

        resultados_dict[f"{metric}_mean"] = np.mean(values)
        resultados_dict[f"{metric}_SD"] = np.std(values, ddof=1)
        resultados_dict[f"{metric}_min"] = np.min(values)
        resultados_dict[f"{metric}_max"] = np.max(values)

    fila_nueva = pd.DataFrame(resultados_dict, index=[metrica])

    # Si el archivo ya existe → actualizar
    if os.path.exists(excel_path):
        df_existente = pd.read_excel(excel_path, index_col=7)

        # Si el indicador ya existe → reemplazar
        df_existente.loc[metrica] = fila_nueva.loc[metrica]

        df_existente.to_excel(excel_path)

    else:
        fila_nueva.to_excel(excel_path)

print(f"\nResultados guardados/actualizados en: {excel_path}")



Resultados guardados/actualizados en: resultados_lstm.xlsx
